In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph,START,END

from pydantic import BaseModel,Field
from typing import Literal
from langchain.agents import create_agent
from langchain_tavily import TavilySearch



In [3]:
class flowst(BaseModel):
    question:str=Field(description="User Input")
    category:Literal['coding','search','weather']=Field(default='search')
    answer:str=Field(default="")

In [4]:
class QuestionCat(BaseModel):
    category:Literal['coding','search','weather']=Field(default='search',description="Question Category")

In [5]:
import os
key1=os.getenv("GroqKEY")
llm1 = ChatGroq(model="openai/gpt-oss-20b", groq_api_key=key1)

In [6]:
def checques(state: flowst) -> flowst:
    # Use method="json_mode" instead of tool calling
    str_llm = llm1.with_structured_output(QuestionCat, method="json_mode")
    
    prompt = (
        f"Classify the following question into exactly one category: 'coding', 'search', or 'weather'.\n"
        f"Return a JSON object with key 'category'.\n"
        f"Question: {state.question}"
    )
    
    res = str_llm.invoke(prompt)
    print(res)
    state.category = res.category
    return state

In [7]:
fst=flowst(question="What is the sppu")
checques(fst)


category='search'


flowst(question='What is the sppu', category='search', answer='')

In [8]:
def routes(state:flowst) -> Literal['coding','search','weather']:
    return state.category


In [9]:
key2=os.getenv("TAVILY_API_KEY")
def get_agent():
    llm = ChatGroq(model="openai/gpt-oss-20b", groq_api_key=key1, streaming=True)
    tavily_search_tool = TavilySearch(
        max_results=5,
        topic="general",
        tavilly_api_key=key2,
        max_tokens=2048
    )
    memory = InMemorySaver()
    
    
    def weather(city: str) -> str:
        """Fetch the real-time weather and temperature for a given city using web search."""
        search_query = f"current weather and temperature in {city}"
        search_results = tavily_search_tool.invoke(search_query)
        return f"Live search results for {city} weather: {search_results}"

    agent_instance = create_agent(
        model=llm,
        tools=[tavily_search_tool,weather],
        system_prompt=(
            "You are an expert AI research assistant with live web search capabilities via Tavily. "
            "Provide accurate, comprehensive, and well-structured answers using Markdown formatting."
        ),
        checkpointer=memory
    )
    return agent_instance

In [10]:
# 1. Initialize agent once
agent = get_agent()

# 2. Node definitions
def coding(state: flowst) -> flowst:
    # Coding doesn't need search, LLM is faster
    res = llm1.invoke(f"You are a coding expert. Answer this question:\n{state.question}")
    state.answer = res.content
    return state

def search(state: flowst) -> flowst:
    # Use agent with Tavily search
    res = agent.invoke({"messages": [{"role": "user", "content": state.question}]})
    state.answer = res["messages"][-1].content
    return state

def weather(state: flowst) -> flowst:
    # Use agent with weather tool
    res = agent.invoke({"messages": [{"role": "user", "content": state.question}]})
    state.answer = res["messages"][-1].content
    return state

In [11]:
gph=StateGraph(flowst)

gph.add_node("Checking Question",checques)
gph.add_node("coding",coding)
gph.add_node("search",search)
gph.add_node("weather",weather)


gph.add_edge(START,"Checking Question")
gph.add_conditional_edges("Checking Question",routes)
gph.add_edge("coding",END)
gph.add_edge("weather",END)
gph.add_edge("search",END)

graph=gph.compile()


In [101]:
# from IPython.display import Image
# Image(graph.get_graph().draw_mermaid_png())


In [12]:
res=graph.invoke({"question":"Tell me about lenovo idapad gaming series"})
print(res["answer"])

category='search'


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-20b` in organization `org_01m66x7p98envbxw4gc4hcchz0` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 9805, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}